# Parsing - What's in this citation?

German legal citations have a lot of structural information into a short string.
This notebook walks through what `parse_reference` extracts and why it matters.

In [1]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".."], check=True)
sys.path.insert(0, str(__import__("pathlib").Path("..").resolve()))

from bundesrecht import parse_reference

## A simple citation

Start with the most common form - a paragraph reference with an Absatz and Nummer.

In [2]:
ref = parse_reference("§ 2 Abs. 1 Nr. 1 UrhG")
print(str(ref))

§ 2 Abs. 1 Nr. 1 UrhG


In [3]:
para = ref.paragraphs[0]
print(para.paragraph)  # the paragraph number

2


In [4]:
for sr in para.sub_refs:
    print(f"{sr.level}: {sr.number}")

Abs: 1
Nr: 1


## Article references

The Grundgesetz and EU regulations use `Art.` instead of `§`.
The parser sets `is_art=True` and handles letter suffixes correctly.

In [5]:
ref = parse_reference("Art. 20 Abs. 3 GG")
print(f"is_art={ref.is_art}, law={ref.law}")

is_art=True, law=GG


In [6]:
# Art. 45a - the letter is part of the article number, not a Buchstabe
ref = parse_reference("Art. 45a GG")
print(f"paragraph={ref.paragraphs[0].paragraph}, sub_refs={ref.paragraphs[0].sub_refs}")

paragraph=45a, sub_refs=[]


## Multi-level sub-references

Citations often come down several levels: Absatz, then Nummer, then Buchstabe.

In [7]:
ref = parse_reference("§ 81 Abs. 1 Nr. 1 Buchst. a BGB")
for sr in ref.paragraphs[0].sub_refs:
    print(str(sr))

Abs. 1
Nr. 1
Buchst. a


## Multi-paragraph citations

A single citation string can reference multiple paragraphs - common with `§§`.

In [8]:
ref = parse_reference("§§ 46 Abs. 2 ArbGG, 91 Abs. 1 ZPO")
for para in ref.paragraphs:
    print(f"§ {para.paragraph} {para.sub_refs} -> law context: {ref.law}")

§ 46 [SubReference(level='Abs', number='2', range_end=None)] -> law context: ArbGG , 91 Abs. 1 ZPO


## Continuation markers - f. and ff.

`f.` (und folgende) means that paragraph and the next one.
`ff.` (und fortfolgende) means that paragraph and several following ones.
The parser captures these as flags on the `ParagraphRef`.

In [9]:
ref = parse_reference("§ 312 f. BGB")
print(f"is_f={ref.paragraphs[0].is_f}, is_ff={ref.paragraphs[0].is_ff}")

is_f=True, is_ff=False


In [10]:
ref = parse_reference("§ 312 ff. BGB")
print(f"is_f={ref.paragraphs[0].is_f}, is_ff={ref.paragraphs[0].is_ff}")

is_f=False, is_ff=True


## No-space citations

In older court decisions and OCR'd documents, the space between `§` and the
paragraph number is often missing. The parser handles this correctly.

In [11]:
ref = parse_reference("§433 Abs. 1 BGB")
print(f"paragraph={ref.paragraphs[0].paragraph}, law={ref.law}")

paragraph=433, law=BGB


## Range citations

A `bis` range sets `range_end` on the `ParagraphRef` - the parser captures
the boundary but leaves expansion to the normaliser.

Note: `-` and `und` between paragraph numbers are handled by the normaliser, 
not the parser.

In [12]:
ref = parse_reference("§§ 12 bis 15 BGB")
para = ref.paragraphs[0]
print(f"paragraph={para.paragraph}, range_end={para.range_end}")

paragraph=12, range_end=15
